In [1]:
# Find the Sutiable Learning Rate
    
# T5
for task in ["seqc", "tokenc"]:
    for language in ["swa"]: # , "eng", "ewe"
        for model in [
            "google/mt5-large", # T5
            "castorini/afriteva_v2_large", # T5
        ]:
            for lr in [5e-5, 1e-4, 2.5e-4, 5e-4, 7.5e-4, 1e-3]:
                print(f"python -m mlds.experiments.finetune_seq2seq {language} {task} {model} --lr {lr}")

python -m mlds.experiments.finetune_seq2seq swa seqc google/mt5-large --lr 5e-05
python -m mlds.experiments.finetune_seq2seq swa seqc google/mt5-large --lr 0.0001
python -m mlds.experiments.finetune_seq2seq swa seqc google/mt5-large --lr 0.00025
python -m mlds.experiments.finetune_seq2seq swa seqc google/mt5-large --lr 0.0005
python -m mlds.experiments.finetune_seq2seq swa seqc google/mt5-large --lr 0.00075
python -m mlds.experiments.finetune_seq2seq swa seqc google/mt5-large --lr 0.001
python -m mlds.experiments.finetune_seq2seq swa seqc castorini/afriteva_v2_large --lr 5e-05
python -m mlds.experiments.finetune_seq2seq swa seqc castorini/afriteva_v2_large --lr 0.0001
python -m mlds.experiments.finetune_seq2seq swa seqc castorini/afriteva_v2_large --lr 0.00025
python -m mlds.experiments.finetune_seq2seq swa seqc castorini/afriteva_v2_large --lr 0.0005
python -m mlds.experiments.finetune_seq2seq swa seqc castorini/afriteva_v2_large --lr 0.00075
python -m mlds.experiments.finetune_seq2se

In [3]:
import json
from collections import defaultdict
import pandas as pd

logfile = "data/results/finetune/log.jsonl"

table = defaultdict(lambda: defaultdict(dict))
for line in open(logfile):
    data = json.loads(line)
    model = data["model"]
    task = data["task"]
    lr = data["lr"]
    col = "accuray"
    for key in data:
        if "f1" in key:
            col = key
            break
    else:
        for key in data:
            if "accuracy" in key:
                col = key
                break

    if "validation_f1" in data:
        table[model][task][lr] = data[col]
    else:
        table[model][task][lr] = data[col]

# find the best model for each model
padding = lambda x: x + " " * (30 - len(x))
lrs = [1e-5, 2e-5, 3e-5, 5e-5, 1e-4, 2.5e-4, 3e-4, 5e-4, 7.5e-4, 1e-3]
lrs = [1e-5, 2e-5, 3e-5, 5e-5, 1e-4, 2e-4, 3e-4, 5e-4]

# Decoder		Llama-3.2-1B-Instruct
# Decoder		Llama-3.2-3B-Instruct
# Other		NLLB-LLM2Vec-Meta-Llama-31-8B-Instruct-mntp-unsup-simcse
# Encoder		afriberta_v2_large
# Encoder-Decoder		afriteva_v2_large
# Encoder		afro-xlmr-large
# Encoder		afro-xlmr-large-76L
# Decoder		gemma-2-2b-it
# Encoder-Decoder		mt5-large
# Encoder		xlm-roberta-large

model_types = {
    "Decoder": {
        "Llama-3.2-1B-Instruct": "meta-llama/Llama-3.2-1B-Instruct",
        "Llama-3.2-3B-Instruct": "meta-llama/Llama-3.2-3B-Instruct",
        "gemma-2-2b-it": "google/gemma-2-2b-it",
    },
    "Other": {
        "NLLB-LLM2Vec-Meta-Llama-31-8B-Instruct-mntp-unsup-simcse": "fdschmidt93/NLLB-LLM2Vec-Meta-Llama-31-8B-Instruct-mntp-unsup-simcse",
    },
    "Encoder": {
        "afriberta_v2_large": "castorini/afriberta_v2_large",
        "afro-xlmr-large": "Davlan/afro-xlmr-large",
        "afro-xlmr-large-76L": "Davlan/afro-xlmr-large-76L",
        "xlm-roberta-large": "FacebookAI/xlm-roberta-large",
    },
    "Encoder-Decoder": {
        "afriteva_v2_large": "castorini/afriteva_v2_large",
        "mt5-large": "google/mt5-large",
    },
}

# expand on the task:[seqc, tokenc] and model_types
data = []
for task in ["seqc", "tokenc"]:
    for model_type in model_types:
        for model in model_types[model_type]:
            data.append([model, model_type, task] + [table[model][task].get(lr, -1) for lr in lrs])
            
df = pd.DataFrame(data, columns=["model", "structure", "task"] + [f"lr_{x}" for x in lrs])
seqc = df[df["task"] == "seqc"]
tokenc = df[df["task"] == "tokenc"]

def get_optimal_lr(df):
    means = []
    for lr in lrs:
        df[f"lr_{lr}"] = df[f"lr_{lr}"].replace(-1, None)
        means.append(df[f"lr_{lr}"].mean())

    # df add mean row
    df = pd.concat([df, pd.DataFrame(
        [means], columns=[f"lr_{x}" for x in lrs], index=["mean"]
    )])
    df = pd.concat([df[[f"lr_{x}" for x in lrs]].idxmax(axis=1), df], axis=1)
    return df

processed = df.groupby(["task", "structure"]).apply(get_optimal_lr).drop(columns=["task", "structure"]).fillna(-1)
print(processed.to_csv("lr.csv"))

None


/tmp/ipykernel_3809459/2443876930.py:105: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  processed = df.groupby(["task", "structure"]).apply(get_optimal_lr).drop(columns=["task", "structure"]).fillna(-1)


In [4]:
def table_get(model, task, lr):
    try:
        model = model.split("/")[-1]
        return table[model][task][lr]
    except KeyError:
        return -1

for task in ["seqc", "tokenc"]:
    for language in ["swa"]:
        # Encoder-Decoder
        for model in [
            "google/mt5-large",
            "castorini/afriteva_v2_large",
        ]:
            for lr in [5e-5, 1e-4, 2.5e-4, 5e-4, 7.5e-4, 1e-3]:
                if 0.0 <= table_get(model, task, lr) <= 0.1:
                    break
                if table_get(model, task, lr) == -1:
                    print(f"python -m mlds.experiments.finetune_seq2seq {language} {task} {model} --lr {lr}")

        # Other
        for model in [
            "FacebookAI/xlm-roberta-large",
            "Davlan/afro-xlmr-large",
            "Davlan/afro-xlmr-large-76L",
            "castorini/afriberta_v2_large",
            'fdschmidt93/NLLB-LLM2Vec-Meta-Llama-31-8B-Instruct-mntp-unsup-simcse',
        ]:
            for lr in [1e-5, 2e-5, 3e-5, 5e-5, 1e-4, 3e-4, 5e-4]:
                if 0.0 <= table_get(model, task, lr) <= 0.1 and task == "seqc":
                    break
                if table_get(model, task, lr) == -1:
                    print(f"python -m mlds.experiments.finetune {language} {task} {model} --lr {lr}")

In [1]:
# # !sbatch --gres=gpu:4 -c 24 --mem=128G --partition=long --time=8:00:00  --constraint="lovelace|ampere" ./task/ddp_root.sh 0 4
# !sbatch --gres=gpu:4 -c 24 --mem=128G --partition=long --time=3:00:00  --constraint="lovelace|ampere" ./task/ddp_root.sh 1 4
# !sbatch --gres=gpu:4 -c 24 --mem=128G --partition=long --time=3:00:00  --constraint="lovelace|ampere" ./task/ddp_root.sh 2 4
# !sbatch --gres=gpu:4 -c 24 --mem=128G --partition=long --time=3:00:00  --constraint="lovelace|ampere" ./task/ddp_root.sh 3 4
# !sbatch --gres=gpu:4 -c 24 --mem=128G --partition=long --time=3:00:00  --constraint="lovelace|ampere" ./task/ddp_root.sh 4 4
# !sbatch --gres=gpu:4 -c 24 --mem=128G --partition=long --time=3:00:00  --constraint="lovelace|ampere" ./task/ddp_root.sh 5 4
# !sbatch --gres=gpu:4 -c 24 --mem=128G --partition=long --time=3:00:00 --constraint="lovelace|ampere" ./task/ddp_root.sh 6 4 
# !sbatch --gres=gpu:4 -c 24 --mem=128G --partition=long --time=3:00:00 ./task/ddp_root.sh 7 4
# !sbatch --gres=gpu:l40s:4 -c 24 --mem=128G --partition=long --time=8:00:00 ./task/ddp_root.sh 8 4
# !sbatch --gres=gpu:4 -c 24 --mem=128G --partition=long --time=8:00:00  --constraint="lovelace|ampere" ./task/ddp_root.sh 9 4

!sbatch --gres=gpu:h100:4 -c 24 --mem=128G --partition=short-unkillable --time=3:00:00 ./task/ddp_root.sh 14 4
!sbatch --gres=gpu:h100:4 -c 24 --mem=128G --partition=short-unkillable --time=3:00:00 ./task/ddp_root.sh 15 4


/bin/bash: /home/mila/h/hao.yu/.conda/envs/rag/lib/libtinfo.so.6: no version information available (required by /bin/bash)
Submitted batch job 5864726
/bin/bash: /home/mila/h/hao.yu/.conda/envs/rag/lib/libtinfo.so.6: no version information available (required by /bin/bash)
Submitted batch job 5864727


In [6]:
# !sbatch --gres=gpu:4 -c 12 --mem=128G --partition=long --time=6:00:00 --constraint="ampere|lovelace" ./task/ddp_root.sh 0 4
# !sbatch --gres=gpu:4 -c 12 --mem=128G --partition=long --time=6:00:00 --constraint="ampere|lovelace" ./task/ddp_root.sh 1 4
# !sbatch --gres=gpu:4 -c 12 --mem=128G --partition=long --time=6:00:00 --constraint="ampere|lovelace" ./task/ddp_root.sh 2 4
# !sbatch --gres=gpu:4 -c 12 --mem=128G --partition=long --time=6:00:00 --constraint="ampere|lovelace" ./task/ddp_root.sh 3 4
# !sbatch --gres=gpu:4 -c 12 --mem=128G --partition=long --time=6:00:00 --constraint="ampere|lovelace" ./task/ddp_root.sh 4 4
!sbatch --gres=gpu:4 -c 12 --mem=128G --partition=long --time=6:00:00 --constraint="ampere|lovelace" ./task/ddp_root.sh 5 4
!sbatch --gres=gpu:4 -c 12 --mem=128G --partition=long --time=6:00:00 --constraint="ampere|lovelace" ./task/ddp_root.sh 6 4
!sbatch --gres=gpu:4 -c 12 --mem=128G --partition=long --time=6:00:00 --constraint="ampere|lovelace" ./task/ddp_root.sh 7 4
!sbatch --gres=gpu:4 -c 12 --mem=128G --partition=long --time=6:00:00 --constraint="ampere|lovelace" ./task/ddp_root.sh 8 4
!sbatch --gres=gpu:4 -c 12 --mem=128G --partition=long --time=6:00:00 --constraint="ampere|lovelace" ./task/ddp_root.sh 9 4


/bin/bash: /home/mila/h/hao.yu/.conda/envs/rag/lib/libtinfo.so.6: no version information available (required by /bin/bash)
Submitted batch job 5768573
/bin/bash: /home/mila/h/hao.yu/.conda/envs/rag/lib/libtinfo.so.6: no version information available (required by /bin/bash)
Submitted batch job 5768574
/bin/bash: /home/mila/h/hao.yu/.conda/envs/rag/lib/libtinfo.so.6: no version information available (required by /bin/bash)
Submitted batch job 5768575
/bin/bash: /home/mila/h/hao.yu/.conda/envs/rag/lib/libtinfo.so.6: no version information available (required by /bin/bash)
Submitted batch job 5768576
/bin/bash: /home/mila/h/hao.yu/.conda/envs/rag/lib/libtinfo.so.6: no version information available (required by /bin/bash)
Submitted batch job 5768577


In [7]:
!sbatch --gres=gpu:4 -c 12 --mem=128G --partition=long --time=6:00:00 --constraint="ampere|lovelace" ./task/root.sh rerun 4


/bin/bash: /home/mila/h/hao.yu/.conda/envs/rag/lib/libtinfo.so.6: no version information available (required by /bin/bash)
Submitted batch job 5768578


In [4]:
# !sbatch --gres=gpu:2 -c 8 --mem=256G --partition=long --time=8:00:00 ./task/root.sh lr 2 --constraint="40gb|80gb|48gb"
!sbatch --gres=gpu:h100:4 -c 24 --mem=128G --partition=short-unkillable --time=3:00:00 ./task/root.sh rerun 4
# !sbatch --gres=gpu:4 -c 12 --mem=128G --partition=long --time=6:00:00 --constraint="ampere|volta|lovelace" ./task/root_ddp.sh 1 4

# !sbatch --gres=gpu:h100:4 -c 24 --mem=128G --partition=short-unkillable --time=3:00:00 ./task/root.sh rerun 4
# !sbatch --gres=gpu:h100:4 -c 24 --mem=128G --partition=short-unkillable --time=3:00:00 ./task/root.sh rerun 4
# !sbatch --gres=gpu:h100:4 -c 24 --mem=128G --partition=short-unkillable --time=3:00:00 ./task/root.sh rerun 4
# !sbatch --gres=gpu:h100:4 -c 24 --mem=128G --partition=short-unkillable --time=3:00:00 ./task/root.sh rerun 4
# !sbatch --gres=gpu:h100:4 -c 24 --mem=128G --partition=short-unkillable --time=3:00:00 ./task/root.sh rerun 4
!sbatch --gres=gpu:4 -c 16 --mem=128G --partition=long --time=24:00:00 ./task/root.sh seed1 4 --constraint="a100l|l40s"
!sbatch --gres=gpu:4 -c 16 --mem=128G --partition=long --time=24:00:00 ./task/root.sh seed2 4 --constraint="a100l|l40s"
!sbatch --gres=gpu:4 -c 16 --mem=128G --partition=long --time=24:00:00 ./task/root.sh seed3 4 --constraint="a100l|l40s"
!sbatch --gres=gpu:4 -c 16 --mem=128G --partition=long --time=24:00:00 ./task/root.sh seed4 4 --constraint="a100l|l40s"
!sbatch --gres=gpu:4 -c 16 --mem=128G --partition=long --time=24:00:00 ./task/root.sh seed5 4 --constraint="a100l|l40s"
# !sbatch --gres=gpu:a100l:4 -c 24 --mem=128G --partition=long --time=24:00:00 ./task/root.sh seed4 4

# !sbatch --gres=gpu:a100l:4 -c 12 --mem=128G --partition=long --time=20:00:00 ./task/root.sh lr 4
!squeue -u $USER
#SBATCH --gpus-per-task=h100:1
#SBATCH --cpus-per-task=6
#SBATCH --mem-per-gpu=128G
#SBATCH --time=3:00:00
#SBATCH --partition=short-unkillable


/bin/bash: /home/mila/h/hao.yu/.conda/envs/rag/lib/libtinfo.so.6: no version information available (required by /bin/bash)
Submitted batch job 5857803
/bin/bash: /home/mila/h/hao.yu/.conda/envs/rag/lib/libtinfo.so.6: no version information available (required by /bin/bash)
Submitted batch job 5857804
/bin/bash: /home/mila/h/hao.yu/.conda/envs/rag/lib/libtinfo.so.6: no version information available (required by /bin/bash)
Submitted batch job 5857805
/bin/bash: /home/mila/h/hao.yu/.conda/envs/rag/lib/libtinfo.so.6: no version information available (required by /bin/bash)
Submitted batch job 5857806
/bin/bash: /home/mila/h/hao.yu/.conda/envs/rag/lib/libtinfo.so.6: no version information available (required by /bin/bash)
Submitted batch job 5857807
/bin/bash: /home/mila/h/hao.yu/.conda/envs/rag/lib/libtinfo.so.6: no version information available (required by /bin/bash)
Submitted batch job 5857808
/bin/bash: /home/mila/h/hao.yu/.conda/envs/rag/lib/libtinfo.so.6: no version information avail

In [9]:
from mlds.data_loader import LANGUAGES
# train: 2240 dev 320 test 640
# mT5/T5

for task in ["seqc", "tokenc"]:
    for language in ["eng"] + LANGUAGES:
        for model in [
            "google/mt5-large",
            "castorini/afriteva_v2_large",
        ]:
            for seed in range(2024, 2029):
                print(f"python -m mlds.experiments.finetune_seq2seq {language} {task} {model} --seed {seed}")


python -m mlds.experiments.finetune_seq2seq eng seqc google/mt5-large --seed 2024
python -m mlds.experiments.finetune_seq2seq eng seqc google/mt5-large --seed 2025
python -m mlds.experiments.finetune_seq2seq eng seqc google/mt5-large --seed 2026
python -m mlds.experiments.finetune_seq2seq eng seqc google/mt5-large --seed 2027
python -m mlds.experiments.finetune_seq2seq eng seqc google/mt5-large --seed 2028
python -m mlds.experiments.finetune_seq2seq eng seqc castorini/afriteva_v2_large --seed 2024
python -m mlds.experiments.finetune_seq2seq eng seqc castorini/afriteva_v2_large --seed 2025
python -m mlds.experiments.finetune_seq2seq eng seqc castorini/afriteva_v2_large --seed 2026
python -m mlds.experiments.finetune_seq2seq eng seqc castorini/afriteva_v2_large --seed 2027
python -m mlds.experiments.finetune_seq2seq eng seqc castorini/afriteva_v2_large --seed 2028
python -m mlds.experiments.finetune_seq2seq amh seqc google/mt5-large --seed 2024
python -m mlds.experiments.finetune_seq2seq

In [10]:
# train: 2240 dev 320 test 640
from mlds.data_loader import LANGUAGES

# other models
for task in ["tokenc", "seqc"]: # "joinc"
    for language in LANGUAGES + ["eng"]: # 
        for model in [
            # "FacebookAI/xlm-roberta-large",
            # "Davlan/afro-xlmr-large",
            # "Davlan/afro-xlmr-large-76L",
            # "castorini/afriberta_v2_large",
            'fdschmidt93/NLLB-LLM2Vec-Meta-Llama-31-8B-Instruct-mntp-unsup-simcse',
            # "meta-llama/Llama-3.2-1B-Instruct",
            # "meta-llama/Llama-3.2-3B-Instruct",
            # "google/gemma-2-2b-it",
        ]:
            # for seed in range(2024, 2029):
            #     print(f"python -m mlds.experiments.finetune {language} {task} {model} --seed {seed}")
            print(f"python -m mlds.experiments.finetune {language} {task} {model}")

    print("###"*10)

python -m mlds.experiments.finetune amh tokenc fdschmidt93/NLLB-LLM2Vec-Meta-Llama-31-8B-Instruct-mntp-unsup-simcse
python -m mlds.experiments.finetune ewe tokenc fdschmidt93/NLLB-LLM2Vec-Meta-Llama-31-8B-Instruct-mntp-unsup-simcse
python -m mlds.experiments.finetune hau tokenc fdschmidt93/NLLB-LLM2Vec-Meta-Llama-31-8B-Instruct-mntp-unsup-simcse
python -m mlds.experiments.finetune ibo tokenc fdschmidt93/NLLB-LLM2Vec-Meta-Llama-31-8B-Instruct-mntp-unsup-simcse
python -m mlds.experiments.finetune kin tokenc fdschmidt93/NLLB-LLM2Vec-Meta-Llama-31-8B-Instruct-mntp-unsup-simcse
python -m mlds.experiments.finetune lin tokenc fdschmidt93/NLLB-LLM2Vec-Meta-Llama-31-8B-Instruct-mntp-unsup-simcse
python -m mlds.experiments.finetune lug tokenc fdschmidt93/NLLB-LLM2Vec-Meta-Llama-31-8B-Instruct-mntp-unsup-simcse
python -m mlds.experiments.finetune orm tokenc fdschmidt93/NLLB-LLM2Vec-Meta-Llama-31-8B-Instruct-mntp-unsup-simcse
python -m mlds.experiments.finetune sna tokenc fdschmidt93/NLLB-LLM2Vec-

### SFT LLMs

In [ ]:
# Generate split datasets
# !python src/mlds/utils/sft_data_generator.py

# Train token classification model
!llamafactory-cli train task/config/llama3_tokenc_sft.yaml

# Train sequence classification model  
!llamafactory-cli train task/config/llama3_seqc_sft.yaml

/bin/bash: /home/mila/h/hao.yu/.conda/envs/rag/lib/libtinfo.so.6: no version information available (required by /bin/bash)
^C
Traceback (most recent call last):
  File "/home/mila/h/hao.yu/.conda/envs/rag/bin/llamafactory-cli", line 5, in <module>
    from llamafactory.cli import main
  File "/home/mila/h/hao.yu/multilingual/Multilingual-Dataset/external/LLaMA-Factory/src/llamafactory/__init__.py", line 44, in <module>
    from .extras.env import VERSION
  File "/home/mila/h/hao.yu/multilingual/Multilingual-Dataset/external/LLaMA-Factory/src/llamafactory/extras/env.py", line 22, in <module>
    import peft
  File "/home/mila/h/hao.yu/.conda/envs/rag/lib/python3.10/site-packages/peft/__init__.py", line 22, in <module>
    from .auto import (
  File "/home/mila/h/hao.yu/.conda/envs/rag/lib/python3.10/site-packages/peft/auto.py", line 32, in <module>
    from .mapping import MODEL_TYPE_TO_PEFT_MODEL_MAPPING
  File "/home/mila/h/hao.yu/.conda/envs/rag/lib/python3.10/site-packages/peft/mapp

In [2]:
# !sbatch --gres=gpu:2 -c 8 --mem=256G --partition=long --time=8:00:00 ./task/root.sh lr 2 --constraint="40gb|80gb|48gb"
# !sbatch --gres=gpu:h100:4 -c 24 --mem=128G --partition=short-unkillable --time=3:00:00 ./task/root.sh rerun 4
# !sbatch --gres=gpu:4 -c 12 --mem=128G --partition=long --time=6:00:00 --constraint="ampere|volta|lovelace" ./task/root_ddp.sh 1 4

# !sbatch --gres=gpu:h100:4 -c 24 --mem=128G --partition=short-unkillable --time=3:00:00 ./task/root.sh rerun 4
# !sbatch --gres=gpu:h100:4 -c 24 --mem=128G --partition=short-unkillable --time=3:00:00 ./task/root.sh rerun 4
# !sbatch --gres=gpu:h100:4 -c 24 --mem=128G --partition=short-unkillable --time=3:00:00 ./task/root.sh rerun 4
# !sbatch --gres=gpu:h100:4 -c 24 --mem=128G --partition=short-unkillable --time=3:00:00 ./task/root.sh rerun 4
# !sbatch --gres=gpu:h100:4 -c 24 --mem=128G --partition=short-unkillable --time=3:00:00 ./task/root.sh rerun 4
# !sbatch --gres=gpu:4 -c 24 --mem=128G --partition=long --time=24:00:00 ./task/root.sh seed3 4 --constraint="a100l|l40s"
# !sbatch --gres=gpu:4 -c 24 --mem=128G --partition=long --time=24:00:00 ./task/root.sh seed3 4
# !sbatch --gres=gpu:4 -c 24 --mem=128G --partition=long --time=24:00:00 ./task/root.sh seed4 4 --constraint="a100l|l40s"
# !sbatch --gres=gpu:a100l:4 -c 24 --mem=128G --partition=long --time=24:00:00 ./task/root.sh seed4 4

!sbatch --gres=gpu:4 -c 12 --mem=128G --partition=long --constraint="ampere|lovelace" --time=6:00:00 ./task/ddp_root.sh 10 4
!sbatch --gres=gpu:4 -c 12 --mem=128G --partition=long --constraint="ampere|lovelace" --time=6:00:00 ./task/ddp_root.sh 11 4
!sbatch --gres=gpu:a100l:4 -c 12 --mem=128G --partition=long --time=18:00:00 ./task/ddp_root.sh 12 4
!sbatch --gres=gpu:a100l:4 -c 12 --mem=128G --partition=long --time=18:00:00 ./task/ddp_root.sh 13 4
!squeue -u $USER
#SBATCH --gpus-per-task=h100:1
#SBATCH --cpus-per-task=6
#SBATCH --mem-per-gpu=128G
#SBATCH --time=3:00:00
#SBATCH --partition=short-unkillable


/bin/bash: /home/mila/h/hao.yu/.conda/envs/rag/lib/libtinfo.so.6: no version information available (required by /bin/bash)
Submitted batch job 5765098
/bin/bash: /home/mila/h/hao.yu/.conda/envs/rag/lib/libtinfo.so.6: no version information available (required by /bin/bash)
Submitted batch job 5765099
/bin/bash: /home/mila/h/hao.yu/.conda/envs/rag/lib/libtinfo.so.6: no version information available (required by /bin/bash)
Submitted batch job 5765100
/bin/bash: /home/mila/h/hao.yu/.conda/envs/rag/lib/libtinfo.so.6: no version information available (required by /bin/bash)
Submitted batch job 5765101
/bin/bash: /home/mila/h/hao.yu/.conda/envs/rag/lib/libtinfo.so.6: no version information available (required by /bin/bash)
   JOBID     USER    PARTITION           NAME  ST START_TIME             TIME NODES CPUS TRES_PER_N MIN_MEM NODELIST (REASON) COMMENT
 5765101   hao.yu         long   multilingual  PD N/A                    0:00     1   12 gres/gpu:a    128G  (Priority) (null)
 5765100   h